# Solution B2 · Regression random forest on house prices

**Type**: Implement with scikit-learn

## Objective

Compare on California the linear regression, the geographic KNN (M5), an optimized tree (M7) and the random forest, looking at how the error changes with `n_estimators`, `max_features` and `min_samples_leaf`. Then measure the feature importance by permutation and the cost in time and memory.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import pickle
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

X, y = ds.california()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
error = lambda m, cols=None: mean_squared_error(y_te, m.fit(X_tr if cols is None else X_tr[cols], y_tr).predict(X_te if cols is None else X_te[cols]), squared=False)

## Commented solution

In [ ]:
rmse = {
    "linear": error(LinearRegression()),
    "geo knn": error(KNeighborsRegressor(5), ["Latitude", "Longitude"]),
    "tree": error(DecisionTreeRegressor(min_samples_leaf=20, random_state=0)),
    "forest": error(RandomForestRegressor(200, max_features=0.33, min_samples_leaf=3, random_state=0, n_jobs=-1)),
}
print({k: round(v, 3) for k, v in rmse.items()})

In [ ]:
oob_table = pd.DataFrame(index=[0.2, 0.33, 0.5, 1.0], columns=[1, 3, 10], dtype=float)
for mf in oob_table.index:
    for leaf in oob_table.columns:
        f = RandomForestRegressor(100, max_features=mf, min_samples_leaf=leaf, oob_score=True, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
        oob_table.loc[mf, leaf] = f.oob_score_
pos = np.unravel_index(oob_table.to_numpy().argmax(), oob_table.shape)
best_mf, best_leaf = oob_table.index[pos[0]], oob_table.columns[pos[1]]
print(oob_table.round(4)); print("best:", best_mf, best_leaf)

In [ ]:
final = RandomForestRegressor(200, max_features=best_mf, min_samples_leaf=best_leaf, random_state=0, n_jobs=-1)
t0 = time.time(); final.fit(X_tr, y_tr); t_fit = time.time() - t0
t0 = time.time(); pred = final.predict(X_te); t_predict = time.time() - t0
forest_rmse = mean_squared_error(y_te, pred, squared=False)
size_mb = len(pickle.dumps(final)) / 1e6
print(f"test RMSE {forest_rmse:.3f} | fit {t_fit:.1f} s | predict {t_predict:.2f} s | size {size_mb:.0f} MB")

In [ ]:
result = permutation_importance(final, X_te, y_te, n_repeats=5, random_state=0, scoring="neg_root_mean_squared_error")
perm = pd.Series(result.importances_mean, index=X.columns).sort_values(ascending=False)
top3 = list(perm.index[:3])
print(perm.round(3).to_string())

## Check

In [ ]:
with verify():
    check(set(rmse) == {"linear", "geo knn", "tree", "forest"} and rmse["forest"] < rmse["tree"] - 0.1 and rmse["forest"] < rmse["geo knn"], "the forest beats the tree by over 0.1 and the geographic KNN", "compare the RMSEs")
    check(rmse["forest"] < 0.55 and rmse["forest"] < rmse["linear"] - 0.15, "forest RMSE below 0.55 and over 0.15 better than the linear one", "RandomForestRegressor(200, max_features=0.33, min_samples_leaf=3)")
    check(oob_table.notna().all().all() and best_mf in (0.33, 0.5) and oob_table.loc[1.0].max() < oob_table.loc[0.33].max(), "the best is 0.33 or 0.5; with all the features (bagging) the OOB R2 is worse", "oob_score_ (R2) of RandomForestRegressor")
    check(best_leaf in (1, 3), "with 100 trees the best leaf is small (1 or 3)", "argmax of the table")
    check(forest_rmse < 0.52 and t_fit > t_predict and 20 < size_mb < 2000, "final RMSE below 0.52; training slower than prediction; model of tens or hundreds of MB", "pickle.dumps(model)")
    check(top3[0] == "MedInc" and set(top3) >= {"MedInc"} and len(perm) == 8 and (perm.diff().dropna() <= 1e-12).all(), "the median income is the most important feature; sorted series", "permutation_importance with scoring neg_root_mean_squared_error")
done()

## What you learned

- **The forest is the quality leap over simple models:** RMSE 0.49-0.50 against 0.62 of the optimized tree (M7), 0.53 of the geographic KNN (M5), 0.735 of the linear one. A reduction of about 20% of the error over the tree just by averaging many trees.
- **`max_features` matters, the leaf much less:** with all the features (bagging) the OOB R² is lower; the best is between 0.33 and 0.5. With 100 deep trees (leaf 1-3) the OOB is almost indifferent to the minimum leaf size: the trees are already decorrelated and the average does the rest.
- **The cost is real:** training takes seconds and the model weighs hundreds of MB (200 trees with thousands of nodes each); prediction on 5000 examples takes a fraction of a second but to serve single low-latency requests the depth and the number of trees matter.
- **Permutation importance:** `MedInc` (the income) is by far the most important, followed by the geographic position (`Latitude`, `Longitude`) and `AveOccup`; the others are worth little. It is a reading that does not depend on the units or on the cardinality of the features, unlike the impurity importance.

## Common pitfall

Comparing the forest with an unregularized tree and concluding "the forest is much better": the honest comparison is with the **optimized** tree (here 0.62): the real gain is a fifth of the error, not a third.

## Interview question

<details><summary>What are the most important hyperparameters of a random forest?</summary>

max_features (it controls the correlation between trees and the bias of each) and min_samples_leaf (it regularizes the leaves), then n_estimators as a budget to choose high enough by looking at the OOB error. The maximum depth is rarely needed. For regression the default of max_features
in scikit-learn is 1.0 (all the features: bagging), so it is almost always worth trying smaller values like 0.33.

</details>